In [11]:
import pandas as pd
import numpy as np
import os

def backtest(average_scores_df, stock):
    average_scores_df = average_scores_df[['datetime', 'instrument', 'score']]
    average_scores_df['datetime'] = average_scores_df['datetime'].astype('datetime64[ns]')
    average_scores_df = average_scores_df.sort_values(by=['datetime', 'instrument'], ascending=[True, True])
    average_scores_df = average_scores_df[average_scores_df['instrument'].isin(stock)]

    stock_df = pd.read_csv('/home/liyuante/cikm2024/dataset/zz500_2018_2023_new_2.csv')
    stock_df['dt'] = pd.to_datetime(stock_df['dt'])
    stock_df = stock_df[['kdcode', 'dt', 'close']]
    stock_df = stock_df.rename(columns={'kdcode': 'instrument', 'dt': 'datetime'})
    stock_df = stock_df.sort_values(by=['instrument', 'datetime'])
    stock_df['close_r'] = stock_df['close'] / stock_df['close'].shift(1)
    df_trade = stock_df[stock_df['datetime'] > '2022-12-31']

    x = pd.merge(df_trade, average_scores_df, on=['datetime', 'instrument'], how='outer')
    trade_date_unique = df_trade['datetime'].unique()
    df_return = pd.DataFrame()

    for date in trade_date_unique:
        x_day = x[x['datetime'] == date]
        r_day = x_day.nlargest(10, columns='score').close_r.mean()
        r_day = r_day - 1
        b = {'datetime': date, 'daily_return': r_day}
        df_return = df_return.append(b, ignore_index=True)

    portfolio_df_performance = df_return.set_index(['datetime'])

    alpha_df_performance = pd.DataFrame()
    alpha_df_performance['portfolio_daily_return'] = portfolio_df_performance['daily_return']
    alpha_df_performance['portfolio_net_value'] = (alpha_df_performance['portfolio_daily_return'] + 1).cumprod()

    net_value_columns = ['portfolio_net_value']

    alpha_statistics_df = pd.DataFrame(index=alpha_df_performance[net_value_columns].columns,
                                        columns=["年化收益", "年化波动率", "最大回撤率", "夏普率", "Calmar", "IR"])

    alpha_df_performance.index = pd.to_datetime(alpha_df_performance.index)
    alpha_statistics_df.loc[:, "年化收益"] = np.mean((alpha_df_performance[net_value_columns].tail(1)) ** (252 / len(alpha_df_performance)) - 1)
    alpha_statistics_df.loc[:, "年化波动率"] = np.std(alpha_df_performance[net_value_columns] / alpha_df_performance[net_value_columns].shift(1) - 1) * np.sqrt(252)
    alpha_statistics_df.loc[:, "最大回撤率"] = np.min((alpha_df_performance[net_value_columns] - alpha_df_performance[net_value_columns].cummax()) / alpha_df_performance[net_value_columns].cummax())
    alpha_statistics_df.loc[:, "夏普率"] = alpha_statistics_df["年化收益"] / alpha_statistics_df["年化波动率"]
    alpha_statistics_df.loc[:, "Calmar"] = alpha_statistics_df["年化收益"] / abs(alpha_statistics_df["最大回撤率"])
    alpha_statistics_df.loc[:, "IR"] = np.mean(alpha_df_performance[net_value_columns] / alpha_df_performance[net_value_columns].shift(1) - 1) * np.sqrt(252) / np.std(alpha_df_performance[net_value_columns] / alpha_df_performance[net_value_columns].shift(1) - 1)

    return alpha_statistics_df

def get_merged_score_df(base_path, prediction_index):
    score_dfs = []
    prediction_path = os.path.join(base_path, f'prediction_{prediction_index}')
    
    for j in range(5):
        folder_path = os.path.join(prediction_path, str(j))
        all_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.csv')]
        
        for file in all_files:
            df = pd.read_csv(file)
            score_dfs.append(df)
    
    # 合并所有score_dfs并取均值
    merged_df = pd.concat(score_dfs).groupby(['dt', 'kdcode']).mean().reset_index()
    merged_df.rename(columns={'kdcode': 'instrument', 'dt' : 'datetime'}, inplace=True)
    return merged_df

# 主程序
base_path = '/home/liyuante/2_20240707_csi500_0.6_5_10'
d = pd.read_csv('/home/liyuante/cikm2024/dataset/zz500_2018_2023_new_2.csv')
stock = d['kdcode'].unique()

results = []

for i in range(20):
    average_scores_df = get_merged_score_df(base_path, i)
    result = backtest(average_scores_df, stock)
    result = result[["年化收益", "年化波动率", "最大回撤率", "夏普率", "Calmar", "IR"]]  # 只保留6个指标
    result['folder'] = i
    results.append(result)

# 显示结果
final_df = pd.concat(results).reset_index(drop=True)
print(final_df)


        年化收益     年化波动率     最大回撤率       夏普率    Calmar        IR  folder
0  -0.069002  0.152553 -0.187111 -0.452314 -0.368774 -0.484642       0
1   0.057584  0.161251 -0.199154  0.357107  0.289143  0.293835       1
2  -0.107490  0.152382 -0.291969 -0.705395 -0.368155 -0.815136       2
3   0.035023  0.157904 -0.154747  0.221800  0.226326  0.196183       3
4  -0.107762  0.169181 -0.247650 -0.636963 -0.435138 -0.617507       4
5  -0.231918  0.166635 -0.292575 -1.391772 -0.792680 -1.531950       5
6  -0.134795  0.172372 -0.265385 -0.782004 -0.507924 -0.832940       6
7   0.107285  0.162460 -0.186089  0.660379  0.576526  0.639120       7
8  -0.076600  0.154393 -0.217719 -0.496138 -0.351831 -0.496758       8
9   0.087981  0.171790 -0.195823  0.512141  0.449287  0.450647       9
10 -0.004307  0.154537 -0.183094 -0.027873 -0.023525 -0.107181      10
11  0.132934  0.171282 -0.119193  0.776113  1.115287  0.694460      11
12 -0.115543  0.162350 -0.244673 -0.711689 -0.472232 -0.705033      12
13  0.

In [2]:
final_df

,年化收益,年化波动率,最大回撤率,夏普率,Calmar,IR,folder
0,0.490968,0.206509,-0.136328,2.377463,3.601368,2.003995,0
1,0.366611,0.220917,-0.108049,1.659495,3.393012,1.490871,1
2,0.272061,0.196657,-0.105733,1.383430,2.573083,1.423053,2
3,0.436592,0.201203,-0.083286,2.169911,5.242065,2.005087,3
4,0.279814,0.215489,-0.169037,1.298504,1.655342,1.347084,4
5,0.198576,0.223120,-0.168900,0.889996,1.175698,1.006126,5
6,0.160267,0.198278,-0.111571,0.808298,1.436456,0.948932,6
7,0.163988,0.202271,-0.110098,0.810735,1.489466,0.942290,7
8,0.559737,0.215636,-0.104841,2.595742,5.338918,2.138518,8
9,0.610354,0.218321,-0.103341,2.795676,5.906200,2.334862,9
